# How is land-cover area calculated?

Use area-weighted aggregation for a categorical map.

**Task** · Calculate WorldCover class areas and water, built-up and tree-cover fractions.

Run from top to bottom. Predict the result before changing code, then compare it with the output.

## Before you start

Use a CPU runtime. You need a registered Earth Engine Cloud project and permission to use it. See the [setup guide](https://github.com/steelpl/lecture/blob/main/CE776-climate-change-carbon-water-cycle/GEE_SETUP.md). Authenticate with your own account before querying data.

In [ ]:
%pip -q install earthengine-api==1.7.52 geemap==0.39.0

In [ ]:
import ee
import geemap

PROJECT_ID = ""  # Your Cloud project ID registered for Earth Engine
if not PROJECT_ID.strip():
    raise ValueError("Enter your registered Google Cloud project ID in PROJECT_ID")
ee.Authenticate()
ee.Initialize(project=PROJECT_ID)

In [ ]:
roi = ee.Geometry.Rectangle([126.80, 37.43, 127.15, 37.70], geodesic=False)

In [ ]:
land = ee.ImageCollection('ESA/WorldCover/v100').first().select('Map').clip(roi)
area_image = ee.Image.pixelArea().divide(1e6).rename('area_km2').addBands(land)
areas = area_image.reduceRegion(ee.Reducer.sum().group(groupField=1,groupName='class'),roi,10,maxPixels=20_000_000)

In [ ]:
groups = areas.getInfo().get('groups',[])
class_names = {10:'Tree cover',20:'Shrubland',30:'Grassland',40:'Cropland',50:'Built-up',60:'Bare/sparse',70:'Snow/ice',80:'Permanent water',90:'Herbaceous wetland',95:'Mangroves',100:'Moss/lichen'}
total = sum(g['sum'] for g in groups)
for g in groups:
    print(class_names.get(g['class'],g['class']), round(g['sum'],3), 'km2', round(100*g['sum']/total,2), '%')

In [ ]:
Map = geemap.Map()
Map.centerObject(roi,10)
classes = [10,20,30,40,50,60,70,80,90,95,100]
palette = ['006400','ffbb22','ffff4c','f096ff','fa0000','b4b4b4','f0f0f0','0064c8','0096a0','00cf75','fae6a0']
Map.addLayer(land.remap(classes, list(range(11))), {'min':0,'max':10,'palette':palette}, 'WorldCover 2020')
Map

Explain whether averaging class numbers can describe land cover. Check whether the fraction denominator is the entire administrative area or the valid classified portion of the selected region.

**Takeaway** · Classes are unordered labels. Aggregate area with `pixelArea`. Differences between the 2020 and 2021 editions can reflect algorithm changes as well as actual land-cover change.

## Data and interpretation

- [WorldCover 2020 · ESA WorldCover Consortium · CC BY 4.0](https://developers.google.com/earth-engine/datasets/catalog/ESA_WorldCover_v100)

Observation years define fixed examples, independent of the teaching year. The code demonstrates processing methods; inspect actual maps and values after authentication.